In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import argparse
from pathlib import Path
from argparse import Namespace

from sklearn.model_selection import train_test_split

#torch.set_float32_matmul_precision('high')  # allows TF32
#torch.backends.cuda.matmul.allow_tf32 = True
#torch.backends.cudnn.allow_tf32 = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

from train_sm import MakeDataset, create_data_loaders, EarlyStopping
from sm import NeuralNet

args = Namespace(
    data_dir=Path(f"/gpfs/bwfor/work/ws/hd_gy283-my_data/data_vMB"),
    num_batches=200,
    batch_size=1024,
    x_train_mean=None,
    x_train_std=None,
    y_train_mean=None,
    y_train_std=None,
    layer_dims=[7, 60, 60, 60, 1],
    num_epochs=1_000,
    lr=5e-3,
    save_name=Path(f"/home/hd/hd_hd/hd_gy283/kmc_project/models/sm_vMB_1e7"),
)

train_loader, test_loader, x_train_mean, x_train_std, y_train_mean, y_train_std = create_data_loaders(args.data_dir, args.num_batches, args.batch_size)
args.x_mean = x_train_mean
args.x_std = x_train_std
args.y_mean = y_train_mean
args.y_std = y_train_std
print(args)

In [ ]:
model = NeuralNet(args.layer_dims, args.x_mean, args.x_std, args.y_mean, args.y_std).to(device)
#model = torch.compile(model)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
#scheduler = torch.optim.lr_scheduler.StepLR(optimizer=optimizer, step_size=200, gamma=0.1)
#scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=100, T_mult=2, eta_min=1e-6)

train_losses = []
val_losses = []

early_stopping = EarlyStopping(patience=10, min_delta=0.0001)

for epoch in range(1, args.num_epochs+1):

    model.train()
    running_loss = 0.0
    for inputs_batch, targets_batch in train_loader:
        inputs_batch  = inputs_batch.to(device, non_blocking=True).float()
        targets_batch = targets_batch.to(device, non_blocking=True).float()

        optimizer.zero_grad()
        preds = model(inputs_batch)
        preds = (preds - model.y_mean) / model.y_std
        targets_batch = (targets_batch - model.y_mean) / model.y_std
        loss  = criterion(preds, targets_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs_batch.size(0)

    epoch_train_loss = running_loss / len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs_batch, targets_batch in test_loader:
            inputs_batch  = inputs_batch.to(device, non_blocking=True).float()
            targets_batch = targets_batch.to(device, non_blocking=True).float()

            preds = model(inputs_batch)
            preds = (preds - model.y_mean) / model.y_std
            targets_batch = (targets_batch - model.y_mean) / model.y_std
            loss  = criterion(preds, targets_batch)
            val_loss += loss.item() * inputs_batch.size(0)

    epoch_val_loss = val_loss / len(test_loader.dataset)
    #scheduler.step()

    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
    #current_lr = optimizer.param_groups[0]['lr']
    if epoch % 10 == 0:
        print(f"Epoch {epoch:2d}/{args.num_epochs} "
              f"   Train Loss: {epoch_train_loss:.10f}"
              f"   Val Loss: {epoch_val_loss:.10f}", flush=True)
    #early_stopping(epoch_val_loss)
    if early_stopping.early_stop:
        print("[EARLY STOPPING TRIGGERED]")
        break

torch.save({
    "model_state_dict": model.state_dict(),
    "args": vars(args),
    "train_losses": train_losses,
    "val_losses": val_losses
}, f"{args.save_name}.pth")